Step 1: Introduction & System Profiles

This notebook extends Project 1 into a Responsible AI audit using MNIST-style handwritten digit classification context.


In [ ]:
#todo - load digit image data, set up baseline train/test split, and summarize system profile context.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

random_seed = 42

try:
    mnist_dataset = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    feature_matrix = mnist_dataset.data.astype(np.float32)
    target_vector = mnist_dataset.target.astype(int)
    dataset_name = 'MNIST (OpenML 70,000 samples)'
except Exception:
    local_digits = load_digits()
    feature_matrix = local_digits.data.astype(np.float32)
    target_vector = local_digits.target.astype(int)
    dataset_name = 'sklearn digits fallback (MNIST-like proxy)'

X_train, X_test, y_train, y_test = train_test_split(
    feature_matrix,
    target_vector,
    test_size=0.2,
    random_state=random_seed,
    stratify=target_vector,
)

pixel_scaler = StandardScaler()
X_train_scaled = pixel_scaler.fit_transform(X_train)
X_test_scaled = pixel_scaler.transform(X_test)

print('=' * 72)
print('Project 2 Responsible AI Audit - System Profile')
print('=' * 72)
print(f'Dataset source: {dataset_name}')
print(f'Train shape: {X_train.shape} | Test shape: {X_test.shape}')
print(f'Unique classes: {sorted(np.unique(target_vector).tolist())}')

class_distribution = pd.Series(target_vector).value_counts().sort_index()
print('\nClass distribution (digit frequency):')
for digit_class, class_count in class_distribution.items():
    print(f'  Digit {digit_class}: {class_count:>6}')

plt.figure(figsize=(8, 3))
plt.bar(class_distribution.index.astype(str), class_distribution.values, color='slateblue')
plt.title('Digit Class Distribution')
plt.xlabel('Digit class')
plt.ylabel('Sample count')
plt.tight_layout()
plt.show()


Step 2: Technical & Operational Comparison

Compare algorithm behavior under clean inputs and simulated real-world noise to audit robustness trade-offs.


In [ ]:
#todo - compare baseline model behavior across clean vs noisy inputs plus inference timing.
import time
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

sgd_classifier = SGDClassifier(loss='log_loss', random_state=random_seed)
rf_classifier = RandomForestClassifier(n_estimators=200, random_state=random_seed, n_jobs=-1)

sgd_classifier.fit(X_train_scaled, y_train)
rf_classifier.fit(X_train, y_train)

def add_noise(input_matrix, noise_std=0.35):
    noisy_matrix = input_matrix + np.random.normal(0.0, noise_std, size=input_matrix.shape)
    return noisy_matrix

X_test_noisy = add_noise(X_test)
X_test_noisy_scaled = pixel_scaler.transform(X_test_noisy)

comparison_rows = []

for model_name, model_obj, clean_data, noisy_data in [
    ('SGDClassifier', sgd_classifier, X_test_scaled, X_test_noisy_scaled),
    ('RandomForestClassifier', rf_classifier, X_test, X_test_noisy),
]:
    start_time = time.perf_counter()
    clean_predictions = model_obj.predict(clean_data)
    clean_inference_ms = (time.perf_counter() - start_time) * 1000

    start_time = time.perf_counter()
    noisy_predictions = model_obj.predict(noisy_data)
    noisy_inference_ms = (time.perf_counter() - start_time) * 1000

    comparison_rows.append({
        'model_name': model_name,
        'clean_accuracy': accuracy_score(y_test, clean_predictions),
        'noisy_accuracy': accuracy_score(y_test, noisy_predictions),
        'clean_macro_f1': f1_score(y_test, clean_predictions, average='macro'),
        'noisy_macro_f1': f1_score(y_test, noisy_predictions, average='macro'),
        'clean_inference_ms': clean_inference_ms,
        'noisy_inference_ms': noisy_inference_ms,
    })

comparison_frame = pd.DataFrame(comparison_rows)
comparison_frame['accuracy_drop'] = comparison_frame['clean_accuracy'] - comparison_frame['noisy_accuracy']

print('\nTechnical and Operational Comparison Summary')
print(comparison_frame.to_string(index=False, float_format=lambda value: f'{value:0.4f}'))

plt.figure(figsize=(8, 4))
bar_positions = np.arange(len(comparison_frame))
bar_width = 0.35
plt.bar(bar_positions - bar_width / 2, comparison_frame['clean_accuracy'], width=bar_width, label='Clean accuracy')
plt.bar(bar_positions + bar_width / 2, comparison_frame['noisy_accuracy'], width=bar_width, label='Noisy accuracy')
plt.xticks(bar_positions, comparison_frame['model_name'])
plt.ylabel('Accuracy')
plt.title('Model Robustness Under Input Noise')
plt.ylim(0.0, 1.0)
plt.legend()
plt.tight_layout()
plt.show()


Step 3: Explainability & Transparency Audit

Use global and local interpretability checks aligned with LIME/SHAP concepts to inspect decision logic.


In [ ]:
#todo - run global permutation importance and local pixel-attribution inspection with interactive controls.
from sklearn.inspection import permutation_importance
from ipywidgets import interact, IntSlider

rf_permutation = permutation_importance(
    rf_classifier,
    X_test,
    y_test,
    n_repeats=5,
    random_state=random_seed,
    scoring='f1_macro',
)

most_influential_pixels = np.argsort(rf_permutation.importances_mean)[-10:][::-1]
print('Top 10 influential pixels (global importance proxy for SHAP concepts):')
for pixel_rank, pixel_index in enumerate(most_influential_pixels, start=1):
    pixel_value = rf_permutation.importances_mean[pixel_index]
    print(f'  {pixel_rank:>2}. Pixel {pixel_index:>3} -> mean importance {pixel_value:0.6f}')

example_images = X_test.reshape(len(X_test), int(np.sqrt(X_test.shape[1])), int(np.sqrt(X_test.shape[1])))

def show_local_explanation(sample_index):
    sample_pixels = example_images[sample_index]
    predicted_digit = int(rf_classifier.predict(X_test[sample_index:sample_index + 1])[0])
    true_digit = int(y_test[sample_index])

    # LIME-style local reasoning concept: inspect the highest-intensity local regions.
    local_intensity_map = sample_pixels / (sample_pixels.max() + 1e-9)

    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(sample_pixels, cmap='gray_r')
    axes[0].set_title(f'Input image | true={true_digit}')
    axes[0].axis('off')

    axes[1].imshow(local_intensity_map, cmap='inferno')
    axes[1].set_title(f'Local saliency proxy | pred={predicted_digit}')
    axes[1].axis('off')

    plt.suptitle('Transparency audit using LIME/SHAP-style concepts')
    plt.tight_layout()
    plt.show()

interactive_explainer = interact(
    show_local_explanation,
    sample_index=IntSlider(min=0, max=min(len(X_test) - 1, 50), step=1, value=0),
)


Step 4: Bias & Fairness Analysis

Audit class-level fairness indicators across digit categories, including demographic-parity-style checks and equalized-odds proxies.


In [ ]:
#todo - calculate per-class performance parity metrics and identify fairness disparities.
from sklearn.metrics import classification_report, confusion_matrix

rf_test_predictions = rf_classifier.predict(X_test)
print('Classification report (RandomForestClassifier):')
print(classification_report(y_test, rf_test_predictions, digits=4))

confusion_mat = confusion_matrix(y_test, rf_test_predictions, labels=sorted(np.unique(y_test)))

fairness_rows = []
all_digits = sorted(np.unique(y_test))
predicted_distribution = pd.Series(rf_test_predictions).value_counts(normalize=True).sort_index()
true_distribution = pd.Series(y_test).value_counts(normalize=True).sort_index()

for digit_label in all_digits:
    true_positive = confusion_mat[digit_label, digit_label]
    false_negative = confusion_mat[digit_label, :].sum() - true_positive
    false_positive = confusion_mat[:, digit_label].sum() - true_positive
    true_negative = confusion_mat.sum() - true_positive - false_negative - false_positive

    recall_value = true_positive / (true_positive + false_negative + 1e-9)
    false_positive_rate = false_positive / (false_positive + true_negative + 1e-9)
    demographic_parity_gap = predicted_distribution.get(digit_label, 0.0) - true_distribution.get(digit_label, 0.0)

    fairness_rows.append({
        'digit_class': digit_label,
        'recall': recall_value,
        'false_positive_rate': false_positive_rate,
        'demographic_parity_gap': demographic_parity_gap,
    })

fairness_frame = pd.DataFrame(fairness_rows)
print('\nFairness indicator table (class-level parity audit):')
print(fairness_frame.to_string(index=False, float_format=lambda value: f'{value:0.4f}'))

print('\nFairness disparity summary:')
print(f"  Recall range: {fairness_frame['recall'].min():0.4f} to {fairness_frame['recall'].max():0.4f}")
print(f"  FPR range: {fairness_frame['false_positive_rate'].min():0.4f} to {fairness_frame['false_positive_rate'].max():0.4f}")
print(f"  Avg |demographic parity gap|: {fairness_frame['demographic_parity_gap'].abs().mean():0.4f}")

plt.figure(figsize=(9, 4))
plt.plot(fairness_frame['digit_class'], fairness_frame['recall'], marker='o', label='Recall by class')
plt.plot(fairness_frame['digit_class'], fairness_frame['false_positive_rate'], marker='s', label='False-positive rate by class')
plt.title('Equalized-odds style indicators across digit classes')
plt.xlabel('Digit class')
plt.ylabel('Metric value')
plt.ylim(0.0, 1.0)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Step 5: Governance, Mitigation & Technical Redesign

Design governance controls, human-in-the-loop escalation, and regulatory alignment recommendations.


In [ ]:
#todo - implement confidence-based human review workflow and document governance-aligned controls.
rf_probability_matrix = rf_classifier.predict_proba(X_test)
rf_confidence_scores = rf_probability_matrix.max(axis=1)
rf_predicted_labels = rf_probability_matrix.argmax(axis=1)

confidence_threshold = 0.85
auto_decision_mask = rf_confidence_scores >= confidence_threshold
manual_review_mask = ~auto_decision_mask

auto_accuracy = accuracy_score(y_test[auto_decision_mask], rf_predicted_labels[auto_decision_mask]) if auto_decision_mask.any() else np.nan
review_queue_rate = manual_review_mask.mean()

print('Governance and Mitigation Summary')
print('=' * 72)
print(f'Confidence threshold: {confidence_threshold:0.2f}')
print(f'Automated decisions: {auto_decision_mask.sum()} / {len(auto_decision_mask)}')
print(f'Review queue rate (HITL): {review_queue_rate:0.4f}')
print(f'Accuracy on automated decisions: {auto_accuracy:0.4f}')

regulatory_alignment = {
    'NIST AI RMF': 'Map/Measure/Manage lifecycle with drift checks and fairness monitoring.',
    'EU AI Act (high-risk principles)': 'Traceable decision logs, human oversight, and post-market monitoring.',
    'OECD AI Principles': 'Transparency, robustness, accountability, and inclusive impact review.'
}

print('\nRegulatory alignment checklist:')
for framework_name, framework_action in regulatory_alignment.items():
    print(f'- {framework_name}: {framework_action}')

plt.figure(figsize=(7, 3))
plt.hist(rf_confidence_scores, bins=20, color='darkcyan', alpha=0.85)
plt.axvline(confidence_threshold, color='red', linestyle='--', label=f'Threshold {confidence_threshold:0.2f}')
plt.title('Prediction confidence for HITL routing')
plt.xlabel('Max class probability')
plt.ylabel('Sample count')
plt.legend()
plt.tight_layout()
plt.show()


### Reflective Conclusion

The Project 2 audit shows that high lab performance is not enough for deployment safety. Robustness checks, explainability probes, class-level fairness metrics, and governance controls all contribute to a stronger responsible AI workflow for handwritten digit recognition systems.
